# Phase 1 Data Audit

This notebook inspects Phase 1 ingestion outputs only.

Rules:

1. No downloading.
2. No data cleaning.
3. No production logic.
4. No realised variance, VRP, regimes, or backtests.
5. If any reusable logic is needed, move it into `src/vrp/` and add tests.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## Expected Phase 1 Outputs

In [ ]:
ROOT = Path("..").resolve()

AUDIT_PATH = ROOT / "reports" / "tables" / "data_audit.csv"

PROCESSED_PATHS = {
    "us_vix": ROOT / "data" / "processed" / "us_vix.parquet",
    "us_underlying": ROOT / "data" / "processed" / "us_underlying.parquet",
    "india_vix": ROOT / "data" / "processed" / "india_vix.parquet",
    "india_underlying": ROOT / "data" / "processed" / "india_underlying.parquet",
}

RAW_PATHS = {
    "us_vix_cboe": ROOT / "data" / "raw" / "us_vix_cboe.parquet",
    "us_vix_fred": ROOT / "data" / "raw" / "us_vix_fred.parquet",
    "us_vix_yahoo": ROOT / "data" / "raw" / "us_vix_yahoo.parquet",
    "us_spx_yahoo": ROOT / "data" / "raw" / "us_spx_yahoo.parquet",
    "us_spy_yahoo": ROOT / "data" / "raw" / "us_spy_yahoo.parquet",
    "india_vix_yahoo": ROOT / "data" / "raw" / "india_vix_yahoo.parquet",
    "india_nifty_yahoo": ROOT / "data" / "raw" / "india_nifty_yahoo.parquet",
}

print("Audit path:", AUDIT_PATH)
print("\nProcessed paths:")
for name, path in PROCESSED_PATHS.items():
    print(f"{name:20s} exists={path.exists()} path={path}")

print("\nRaw paths:")
for name, path in RAW_PATHS.items():
    print(f"{name:20s} exists={path.exists()} path={path}")

## Data Audit Table

In [ ]:
audit = pd.read_csv(AUDIT_PATH)
audit

## Validation Failures, If Any

In [ ]:
failures = audit[~audit["validation_status"].astype(str).str.startswith("PASS")]
failures

## Load Processed Datasets

In [ ]:
processed = {name: pd.read_parquet(path) for name, path in PROCESSED_PATHS.items()}

for name, df in processed.items():
    print("=" * 80)
    print(name)
    print(df.shape)
    display(df.head())
    display(df.tail())

## Canonical Schema Check

In [ ]:
CANONICAL_COLUMNS = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "source",
    "market",
    "symbol",
]

schema_rows = []
for name, df in processed.items():
    schema_rows.append(
        {
            "dataset": name,
            "columns_match_exactly": list(df.columns) == CANONICAL_COLUMNS,
            "columns": list(df.columns),
        }
    )

pd.DataFrame(schema_rows)

## Missingness Summary

In [ ]:
missing_rows = []
for name, df in processed.items():
    for column in df.columns:
        missing_rows.append(
            {
                "dataset": name,
                "column": column,
                "n_missing": int(df[column].isna().sum()),
                "missing_fraction": float(df[column].isna().mean()),
            }
        )

missing_summary = pd.DataFrame(missing_rows)
missing_summary[missing_summary["n_missing"] > 0]

## Date Ranges

In [ ]:
date_rows = []
for name, df in processed.items():
    dates = pd.to_datetime(df["date"], errors="coerce")
    date_rows.append(
        {
            "dataset": name,
            "source": df["source"].dropna().unique().tolist(),
            "market": df["market"].dropna().unique().tolist(),
            "symbol": df["symbol"].dropna().unique().tolist(),
            "start_date": dates.min(),
            "end_date": dates.max(),
            "n_rows": len(df),
            "n_unique_dates": df["date"].nunique(),
        }
    )

pd.DataFrame(date_rows)

## Close-Series Visual Check

In [ ]:
for name, df in processed.items():
    plot_df = df.copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"], errors="coerce")
    plot_df = plot_df.sort_values("date")

    plt.figure(figsize=(12, 4))
    plt.plot(plot_df["date"], plot_df["close"])
    plt.title(f"{name}: close")
    plt.xlabel("date")
    plt.ylabel("close")
    plt.grid(True, alpha=0.3)
    plt.show()

## Audit Notes

Manual checks to record after running this notebook:

- Do the four processed datasets exist?
- Does each processed dataset follow canonical schema?
- Are date ranges plausible?
- Are any validation failures present in `data_audit.csv`?
- Are there missing close values?
- Are close plots visually reasonable?
- Are source labels explicit and not silently mixed?